In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
# Preprocesamiento
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve
)

# Modelos
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

EXPLORACIÓN Y DEPURACIÓN DE DATOS

In [ ]:
#PARA EL CONJUNTO DE DATOS
data = pd.read_csv('') #_excel o _json
data.head()
data.tail()
data.sample(5)

data.shape              # (filas, columnas)
data.columns
data.index

data.info()
data.describe()
data.describe(include='object')

data.dtypes


In [ ]:
# VALORES NULOS
data.isnull().sum()
data.isnull().mean() * 100
data.dropna()
data.dropna(axis=1)
data.fillna(0)
data.fillna(data.mean(numeric_only=True))
# ELIMINAR DUPLICADOS
data.duplicated().sum()
data.drop_duplicates()
# CAMBIO DE TIPO
data['col'] = data['col'].astype(int)
data['fecha'] = pd.to_datetime(data['fecha'])


In [ ]:
#PARA UNA COLUMNA
data[''].value_counts() #Numero de veces que se repite un valor
data[''].sort_values(ascending=False) #Ordena los valores de la columna
# Selección de columnas
data['col']
data[['col1', 'col2']]
# Filtrado condicional
data[data['edad'] > 30]
data[(data['edad'] > 30) & (data['salario'] > 2000)]
# Query
data.query("edad > 30 and salario > 2000")
# Ordenar
data.sort_values(by='salario', ascending=False)
# Groupby básico
data.groupby('col').mean(numeric_only=True)
data.groupby('col')['salario'].sum()
# Múltiples agregaciones
data.groupby('col').agg({
    'salario': ['mean', 'max', 'min'],
    'edad': 'median'
})
pd.pivot_table(data, values='salario', index='ciudad', columns='sexo', aggfunc='mean') #tabla pivot

In [55]:
#Operaciones con numpy
# Crear arrays
arr = np.array([1,2,3])
mat = np.array([[1,2],[3,4]])
# Operaciones estadísticas
np.mean(arr)
np.median(arr)
np.std(arr)
np.var(arr)
np.sum(arr)
np.min(arr)
np.max(arr)
# Generación de datos
np.arange(0, 10, 2)
np.linspace(0, 1, 100)
np.random.rand(5)
np.random.randn(5)
np.random.randint(0, 10, 5)
# Álgebra lineal
np.dot(mat, mat)
np.linalg.inv(mat)
np.linalg.det(mat)


In [ ]:
#Visualización con matplotlib
# HISTOGRAMA
plt.hist(data['edad'], bins=20)
plt.title("Distribución Edad")
plt.xlabel("Edad")
plt.ylabel("Frecuencia")
plt.show()
# SCATTER PLOT
plt.scatter(data['edad'], data['salario'], alpha=0.5)
plt.xlabel("Edad")
plt.ylabel("Salario")
plt.show()
# LINE PLOT
plt.plot(data['fecha'], data['ventas'])
plt.xticks(rotation=45)
plt.show()
# BAR PLOT
data['ciudad'].value_counts().plot(kind='bar')
plt.show()

In [ ]:
#Visualizacion con seaborn
# HISTOGRAMA
sns.histplot(data['edad'], kde=True)
# BOXPLOT
sns.boxplot(x='sexo', y='salario', data=data)
# SCATTER + REGRESIÓN
sns.regplot(x='edad', y='salario', data=data)
# COUNT PLOT
sns.countplot(x='sexo', data=data)
# HEATMAP CORRELACIÓN
data_num = data.select_dtypes(include=np.number)
corr = data_num.corr()
sns.heatmap(corr, annot=True, cmap='coolwarm')
plt.show()
# PAIRPLOT
sns.pairplot(data_num)

MACHINE LEARNING

In [ ]:
# Separar variables
X = data.drop('target', axis=1)
y = data['target']
# División train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)
# Escalado (IMPORTANTE para KNN, SVM, Logistic)
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
#KNN
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, y_train)
y_pred_knn = knn.predict(X_test_scaled)
#Arbol de decision
tree = DecisionTreeClassifier(max_depth=5, random_state=42)
tree.fit(X_train, y_train)
y_pred_tree = tree.predict(X_test)
#Bosque aleatorio
rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    random_state=42
)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
#Regresión logística
logreg = LogisticRegression(max_iter=1000)
logreg.fit(X_train_scaled, y_train)
y_pred_log = logreg.predict(X_test_scaled)
#SVM
svm = SVC(kernel='rbf', probability=True)
svm.fit(X_train_scaled, y_train)
y_pred_svm = svm.predict(X_test_scaled)

In [ ]:
#Evaluación de modelos
def evaluar_modelo(y_test, y_pred, nombre):
    print(f"===== {nombre} =====")
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("\nClassification Report:\n", classification_report(y_test, y_pred))

    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d')
    plt.title(f"Matriz Confusión - {nombre}")
    plt.show()
evaluar_modelo(y_test, y_pred_knn, "KNN")
evaluar_modelo(y_test, y_pred_tree, "Decision Tree")
evaluar_modelo(y_test, y_pred_rf, "Random Forest")
evaluar_modelo(y_test, y_pred_log, "Logistic Regression")
evaluar_modelo(y_test, y_pred_svm, "SVM")
